In [1]:
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

C:\Users\kvika\AppData\Local\Temp\ipykernel_4412\3494988372.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [ ]:
# Reading all the pdf's inside the directory
def process_pdfs(pdf_directory):
    all_documents = []
    pdf_dir = Path(pdf_directory)
    pdf_files = list(pdf_dir.glob("*.pdf"))
    print(f"Found {len(pdf_files)} pdf files to process")

    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            all_documents.extend(documents)
            print(f" Loaded {len(documents)} pages")
        except Exception as e:
            print(f" Error: {e}")
    return all_documents

all_pdf_documents = process_pdfs("data")

Found 8 pdf files to process

Processing: 01_Basics_of_Python_Programming.pdf
 Loaded 10 pages

Processing: 02_Introduction_to_Machine_Learning.pdf
 Loaded 9 pages

Processing: 03_Data_Structures_and_Algorithms_Overview.pdf
 Loaded 11 pages

Processing: 04_SQL_Fundamentals.pdf
 Loaded 10 pages

Processing: 05_Pandas_for_Data_Analysis.pdf
 Loaded 10 pages

Processing: 06_Git_and_Version_Control.pdf
 Loaded 9 pages

Processing: 07_Statistics_Essentials_for_Data_Science.pdf
 Loaded 9 pages

Processing: 08_Natural_Language_Processing_Basics.pdf
 Loaded 10 pages


In [ ]:
# Text splitting get into chunks
def split_documents(documents):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1500,
        chunk_overlap=200,
        length_function = len
    )
    split_docs = text_splitter.split_documents(documents)

    # Show example of a chunk
    if split_docs:
        print("\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
    return split_docs

In [4]:
chunks = split_documents(all_pdf_documents)


Example chunk:
Content: STUDY GUIDE  |  PROGRAMMING FUNDAMENTALS
Basics of Python
Programming
Core syntax, data types, control flow, functions and error handling
In this guide
Introduction
1. Setting Up and Running Python
2....


In [5]:
from sentence_transformers import SentenceTransformer
import numpy as np

In [ ]:
class EmbeddingManager:
    # Handles document embedding generation using SentenceTransformer
    def __init__(self, model_name: str ="all-MiniLM-l6-v2"):
        self.model_name = model_name
        self.model = None
        self._load_model()
    
    # Load the SentenceTransformer model
    def _load_model(self):
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print("Model loaded successfully")
        except Exception as e:
            print(f"Error: {e}")
            raise
        
    # Generate embeddings for a list of texts
    def generate_embedding(self, texts:list[str]) -> np.ndarray:
        if not self.model:
            raise ValueError("Model not loaded")
        print(f"Generating embeddings for {len(texts)} texts..")
        embeddings = self.model.encode(texts,show_progress_bar=True)
        return embeddings    

In [7]:
embedding_manager = EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-l6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model loaded successfully


In [ ]:
import uuid
import chromadb
from typing import Any

In [ ]:
class VectorStore:
    # Manages document embeddings in a ChromaDB vector store
    def __init__(self, collection_name: str="pdf_documents",persist_directory: str="vector_store"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    # Initialize ChromaDB client and collection
    def _initialize_store(self):
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory,exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata = {"description": "pdf document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing document in Collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error: {e}")

    # Add documents and their embeddings to the vector store
    def add_documents(self, documents: list[Any], embeddings: np.ndarray):
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        print(f"adding {len(documents)} documents in to vector store.")

        # prepare data for chromadb
        ids=[]
        metadatas=[]
        documents_text=[]
        embeddings_list=[]
        
        for i,(doc, embeddings) in enumerate(zip(documents,embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            # prepare metadata
            metadata = dict(doc.metadata)
            metadatas.append(metadata)
            # Document content
            documents_text.append(doc.page_content)
            # Embedding
            embeddings_list.append(embeddings.tolist())

        # Add to collection
        try:
            self.collection.add(
                ids = ids,
                embeddings = embeddings_list,
                metadatas = metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise
        
Vectorstore=VectorStore()
Vectorstore

Vector store initialized. Collection: pdf_documents
Existing document in Collection: 0


In [10]:
texts = [doc.page_content for doc in chunks]
embeddings = embedding_manager.generate_embedding(texts)
Vectorstore.add_documents(chunks,embeddings)

Generating embeddings for 122 texts..


Batches:   0%|          | 0/4 [00:00<?, ?it/s]

adding 122 documents in to vector store.
Successfully added 122 documents to vector store


# Handles query-based retrieval from the vector store

In [ ]:
class RAGRetriever:
    # Initialize the retriever
    def __init__(self, vector_store:VectorStore, embedding_manager: EmbeddingManager):
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    # Retrieve relevant documents for a query
    def retriever(self,query, top_k: int = 4, score_threshold: float=0.0) -> list[dict[str,Any]]:
        print(f"Retrieving document query: '{query}'")
        print(f"Top k = {top_k}, Score Threshold = {score_threshold}")

        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embedding([query])[0]

        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results = top_k
            )

            # Process results
            retrieved_docs = []
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distance = results['distances'][0]
                ids = results['ids'][0]
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents,metadatas, distance)):
                    similarity_score = 1- distance
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id':doc_id,
                            'content' : document,
                            'metadata':metadata,
                            'similarity_score':similarity_score,
                            'distance':distance,
                            'rank':i+1
                        })
                print(f"Retrieved {len(retrieved_docs)} documents(after filtering)")
            else:
                print("No documents found")
            return retrieved_docs
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retiever = RAGRetriever(Vectorstore, embedding_manager)
rag_retiever

In [12]:
rag_retiever.retriever("Types of Machine Learning")

Retrieving document query: 'Types of Machine Learning'
Top k = 4, Score Threshold = 0.0
Generating embeddings for 1 texts..


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Retrieved 4 documents(after filtering)


[{'id': 'doc_bb353a0c_15',
  'content': 'STUDY GUIDE  |  DATA SCIENCE & AI\nIntroduction to Machine\nLearning\nCore concepts, learning paradigms, algorithms, evaluation and\nworkflow\nIn this guide\nIntroduction\n1. Core Concepts and Terminology\n2. Types of Machine Learning\n3. Common Algorithms\n4. Evaluating Models\n5. Overfitting, Underfitting and the Bias-Variance Trade-off\n6. The Machine Learning Workflow in Practice\nConclusion\nA concise, example-driven introduction for beginners and self-learners.',
  'metadata': {'trapped': '/False',
   'title': 'Introduction to Machine Learning',
   'subject': 'Core concepts, learning paradigms, algorithms, evaluation and workflow',
   'author': 'Study Guide Series',
   'keywords': '',
   'page': 0,
   'creationdate': '2026-09-29T18:56:35+00:00',
   'page_label': '1',
   'creator': '(unspecified)',
   'total_pages': 9,
   'moddate': '2026-09-29T18:56:35+00:00',
   'source': 'data\\02_Introduction_to_Machine_Learning.pdf',
   'producer': 'Re

# Integration Vectordb Context With LLM output

In [ ]:
# RAG pipeline with Groq llm
from langchain_groq import ChatGroq
from dotenv import load_dotenv
load_dotenv()

True

In [ ]:
# Initializing the Groq llm
groq_api_key = os.getenv("GROQ_API_KEY")
llm = ChatGroq(groq_api_key=groq_api_key, model_name="openai/gpt-oss-120b",temperature=0.1,max_tokens=1024)
print("Groq LLM initialized successfully!")

Groq LLM initialized successfully!


In [ ]:
#RAG function: retrieve content + generate response
def rag_retieve(query,retiever,llm,top_k=3):
    # retriever the context
    results=retiever.retriever(query,top_k=top_k)
    context = "\n\n".join(doc['content'] for doc in results) if results else ""
    if not context:
        return "No relevant context found to answer the question"
    
    # generate the answwer using GROQ LLM
    prompt = f"""use the following context to answer the question.
        Context: {context}
        Question: {query}
        Answer: """
    response = llm.invoke([prompt.format(context=context,query=query)])
    return response.content

In [19]:
answer = rag_retieve("Types of Machine Learning",rag_retiever,llm)
print(answer)

Retrieving document query: 'Types of Machine Learning'
Top k = 3, Score Threshold = 0.0
Generating embeddings for 1 texts..


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Retrieved 3 documents(after filtering)
**Types of Machine Learning**

| Category | How it learns | Typical tasks |
|----------|--------------|----------------|
| **Supervised learning** | Learns from data that includes the correct answers (labels). | • **Regression** – predicts a continuous value (e.g., house price, temperature).<br>• **Classification** – predicts a discrete category (e.g., spam vs. not‑spam, digit recognition). |
| **Unsupervised learning** | Learns from data without any labels, discovering structure on its own. | • **Clustering** – groups similar items (e.g., customer segmentation).<br>• **Dimensionality reduction** – compresses many features into a few (e.g., PCA, t‑SNE).<br>• **Anomaly detection** – finds unusual observations (e.g., fraud detection). |
| **Reinforcement learning** | An agent interacts with an environment, takes actions, and receives rewards/penalties; it learns a policy that maximizes long‑term reward. | • Game‑playing agents (e.g., AlphaGo).<br>• 

In [20]:
answer2 = rag_retieve("text preprocessing",rag_retiever,llm) 
print(answer2)

Retrieving document query: 'text preprocessing'
Top k = 3, Score Threshold = 0.0
Generating embeddings for 1 texts..


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Retrieved 1 documents(after filtering)
**Text preprocessing** is the first step in any Natural Language Processing (NLP) workflow. It involves cleaning and transforming raw human language into a structured format that machines can efficiently analyze. Typical preprocessing tasks include:

1. **Normalization** – converting text to a standard form (e.g., lower‑casing, removing punctuation, handling Unicode characters, expanding contractions).
2. **Noise removal** – deleting irrelevant elements such as HTML tags, URLs, email addresses, numbers, or stop words (common words that add little semantic value).
3. **Tokenization** – splitting the text into individual units (tokens) like words, sub‑words, or characters.
4. **Stemming & Lemmatization** – reducing words to their base or root form (e.g., “running” → “run”) to treat morphological variants as the same token.
5. **Handling special cases** – dealing with emojis, hashtags, mentions, or domain‑specific jargon.
6. **Encoding** – converting